In [52]:
# Importing of required items
import pandas as pd
import itertools
import csv

In [ ]:
# Read the file that we will use and assign them variables name
m_seeds = pd.read_csv('MNCAATourneySeeds.csv')
w_seeds = pd.read_csv('WNCAATourneySeeds.csv')
m_games = pd.read_csv('MNCAATourneyCompactResults.csv')
w_games = pd.read_csv('WNCAATourneyCompactResults.csv')
m_teams = pd.read_csv("MTeams.csv")
w_teams = pd.read_csv('WTeams.csv')
sample = pd.read_csv('SampleSubmissionStage1.csv')
sample2 = pd.read_csv('SampleSubmissionStage2.csv')
m_regular = pd.read_csv('MRegularSeasonCompactResults.csv')
w_regular = pd.read_csv('WRegularSeasonCompactResults.csv')

m_all_games = pd.concat([m_games, m_regular])
w_all_games = pd.concat([w_games, w_regular])
print(sample2.head())
print(len(sample2))
print(sample2['ID'].str[:4].unique())
# print(sample.head())
# print(len(sample))

               ID  Pred
0  2026_1101_1102   0.5
1  2026_1101_1103   0.5
2  2026_1101_1104   0.5
3  2026_1101_1105   0.5
4  2026_1101_1106   0.5
132133
<StringArray>
['2026']
Length: 1, dtype: str


In [54]:
# print(m_games['Season'].max())
# print(m_games['Season'].min())

In [ ]:
# Calculating the weights of the men  and women
m_all_games['Weight'] = m_all_games['Season'] - m_games['Season'].min() + 1
w_all_games['Weight'] = w_games['Season'] - w_games['Season'].min() + 1

In [56]:
# Calculating all the wins and lose of the teams then the win rates of the teams
m_win_counts = m_games.groupby('WTeamID')['Weight'].sum() 
m_lose_counts = m_games.groupby('LTeamID')['Weight'].sum()
m_total_counts = (m_win_counts + m_lose_counts).fillna(0)
m_win_rates = (m_win_counts/ m_total_counts).fillna(0.5)
m_win_rates = m_win_rates.replace(0, 0.5)
print(m_win_counts.head())
print(m_win_rates.head())



# Calculating all the win and lose of women, then the win rate of the teams 
w_win_counts = w_games.groupby('WTeamID')['Weight'].sum()
w_lose_counts = w_games.groupby('LTeamID')['Weight'].sum()
# print(m_lose_counts)
# print(m_win_counts)
w_total_counts = (w_win_counts + w_lose_counts).fillna(0)
w_win_rates = (w_win_counts/w_total_counts).fillna(0.5)
w_win_rates = w_win_rates.replace(0.0, 0.5)
# print(total_counts.head())
# print(w_win_rates)



WTeamID
1101      37
1104     636
1106      41
1107      30
1112    1185
Name: Weight, dtype: int64
WTeamID
1101    0.339450
1102    0.500000
1103    0.500000
1104    0.608031
1105    0.500000
Name: Weight, dtype: float64


In [57]:
m_seeds['Seed'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['Seed'] = w_seeds['Seed'].str[1:3].astype(int)
# print(m_seeds.head())
print(w_seeds.head())

   Season  Seed  TeamID
0    1998     1    3330
1    1998     2    3163
2    1998     3    3112
3    1998     4    3301
4    1998     5    3272


In [58]:
# We get the latest seed of each team
m_seeds_latest = m_seeds[m_seeds['Season'] == m_seeds['Season'].max()]
w_seeds_latest = w_seeds[w_seeds['Season'] == w_seeds['Season'].max()] 

print(m_seeds_latest.head())
# print(m_seeds.head())
print(w_seeds_latest.head())
# print(w_seeds.head())
# print(m_seeds['Seed'].unique())
# print(w_seeds['Seed'].unique())


      Season  Seed  TeamID
2558    2025     1    1181
2559    2025     2    1104
2560    2025     3    1458
2561    2025     4    1112
2562    2025     5    1332
      Season  Seed  TeamID
1676    2025     1    3376
1677    2025     2    3181
1678    2025     3    3314
1679    2025     4    3268
1680    2025     5    3104


In [59]:
# Generates the matchups of the men's and women's teams
all_m_matchups = (list(itertools.combinations(m_teams['TeamID'].values, 2)))
all_w_matchups = (list(itertools.combinations(w_teams['TeamID'].values, 2)))
# print(all_m_matchups)
# print(len(all_w_matchups))

In [60]:
# Convert the TeamID from interger so as to combine with m_seed_latest
m_win_rates_df = m_win_rates.reset_index()
m_win_rates_df.columns = ['TeamID', 'WinRate']

w_win_rates_df = w_win_rates.reset_index()
w_win_rates_df.columns = ['TeamID', 'WinRate']

# print(w_win_rates_df.head())
# print(m_win_rates_df.head())

In [61]:
# Calculating the strength
m_seeds_latest['Strength'] = (17 - m_seeds_latest['Seed'])
# Combining the m_seed_latest and m_win_rate_df
m_team_stats = m_teams[['TeamID']].merge(m_win_rates_df, on='TeamID', how='left')
m_team_stats = m_team_stats.merge(m_seeds_latest, on='TeamID', how='left')
# Fill unknown teams with neautral strength and probability
m_team_stats['WinRate'] = m_team_stats['WinRate'].fillna(0.5)
m_team_stats['Strength']  = m_team_stats['Strength'].fillna(9)
m_team_stats['Score'] = m_team_stats['WinRate'] * m_team_stats['Strength']

w_seeds_latest['Strength'] = (17 - w_seeds_latest['Seed'])
w_team_stats = w_teams[['TeamID']].merge(w_win_rates_df, on='TeamID', how='left')
w_team_stats = w_team_stats.merge(w_seeds_latest, on='TeamID', how='left')
w_team_stats['WinRate'] = w_team_stats['WinRate'].fillna(0.5)
w_team_stats['Strength'] = w_team_stats['Strength'].fillna(9)
w_team_stats['Score'] = w_team_stats['WinRate'] * w_team_stats['Strength']
# print(m_team_stats.head(10))
# print(w_team_stats.head(10))
# print(len(m_team_stats))
# print(m_team_stats[m_team_stats['TeamID'] == 1181])

In [62]:
sample_2025 = sample[sample['ID'].str.startswith('2025')]
print(len(sample_2025))

131407


In [63]:
print(len(sample_2025))
print(sample_2025['ID'].str[:4].unique())
print(sample_2025['ID'].duplicated().sum())

131407
<StringArray>
['2025']
Length: 1, dtype: str
0


In [64]:
# Calculating the probability between the two teams
rows = []
for idx, row in sample2.iterrows():
    parts = row['ID'].split('_')
    team1 = int(parts[1])
    team2 = int(parts[2])
    
    if team1 < 3000:
        score1 =m_team_stats[m_team_stats['TeamID'] == team1]['Score'].values[0]
        score2 = m_team_stats[m_team_stats['TeamID'] == team2]['Score'].values[0]
    else:
        score1 = w_team_stats[w_team_stats['TeamID'] == team1]['Score'].values[0]
        score2 = w_team_stats[w_team_stats['TeamID'] == team2]['Score'].values[0]



    propablity = score1 / (score1 + score2)
    match_id = f"2026_{parts[1]}_{parts[2]}"
    rows.append([row['ID'], propablity])

df = pd.DataFrame(rows, columns=['ID', 'Pred'])
df = df.drop_duplicates(subset='ID')

print(len(rows))
print(rows[:3])

132133
[['2026_1101_1102', np.float64(0.40437158469945356)], ['2026_1101_1103', np.float64(0.604355716878403)], ['2026_1101_1104', np.float64(0.2509173895566707)]]


In [65]:
print(sample['ID'].head())
print(sample['ID'].str[:4].unique())

0    2022_1101_1102
1    2022_1101_1103
2    2022_1101_1104
3    2022_1101_1105
4    2022_1101_1106
Name: ID, dtype: str
<StringArray>
['2022', '2023', '2024', '2025']
Length: 4, dtype: str


In [66]:
# print(m_games[m_games['WTeamID'] == 1181]['WTeamID'].count())
# print(m_win_counts[1181])

In [68]:

df = pd.DataFrame(rows, columns=['ID', 'Pred'])

df.to_csv('matchup_probabilities.csv', index=False)
print(df.head())
print(len(df))
print(df.isnull().sum())
print(df['Pred'].max())
print(df['Pred'].min())
print(df[df['ID'].str.contains('_3')].head())
print(df[df['ID'].str.contains('_1')].head())


               ID      Pred
0  2026_1101_1102  0.404372
1  2026_1101_1103  0.604356
2  2026_1101_1104  0.250917
3  2026_1101_1105  0.404372
4  2026_1101_1106  0.927229
132133
ID      0
Pred    0
dtype: int64
0.9934869040560806
0.006443632865759566
                   ID      Pred
66430  2026_3101_3102  0.500000
66431  2026_3101_3103  0.500000
66432  2026_3101_3104  0.463217
66433  2026_3101_3105  0.500000
66434  2026_3101_3106  0.500000
               ID      Pred
0  2026_1101_1102  0.404372
1  2026_1101_1103  0.604356
2  2026_1101_1104  0.250917
3  2026_1101_1105  0.404372
4  2026_1101_1106  0.927229


In [69]:
print(len(df))
print(df.isnull().sum())

132133
ID      0
Pred    0
dtype: int64


In [70]:
print(df.head())
print(df['Pred'].max())
print(df['Pred'].min())


               ID      Pred
0  2026_1101_1102  0.404372
1  2026_1101_1103  0.604356
2  2026_1101_1104  0.250917
3  2026_1101_1105  0.404372
4  2026_1101_1106  0.927229
0.9934869040560806
0.006443632865759566


In [71]:
print(df['ID'].head())
print(df['ID'].str[:4].unique())

0    2026_1101_1102
1    2026_1101_1103
2    2026_1101_1104
3    2026_1101_1105
4    2026_1101_1106
Name: ID, dtype: str
<StringArray>
['2026']
Length: 1, dtype: str


In [72]:
print(df.isnull().sum())
print(df['Pred'].max())
print(df['Pred'].min())

ID      0
Pred    0
dtype: int64
0.9934869040560806
0.006443632865759566


In [73]:
print(len(df))           # Should be 132,133
print(df['ID'].head())   # Should show 2026_...
print(df.isnull().sum()) # Should be 0

132133
0    2026_1101_1102
1    2026_1101_1103
2    2026_1101_1104
3    2026_1101_1105
4    2026_1101_1106
Name: ID, dtype: str
ID      0
Pred    0
dtype: int64
